In [ ]:
import pathlib
import sys

import networkx as nx
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches 
from vizopt import bubblejax

In [ ]:
# Example: Single inclusion tree with leaf nodes (cities) and non-leaf nodes (countries)
# Leaf nodes have fixed "size" attributes, non-leaf nodes have optimizable radii

tree = nx.DiGraph()

# Add leaf nodes (cities) with fixed sizes
cities = {
    "Munich": 10,
    "Berlin": 25,
    "Stuttgart": 5,
    "Frankfurt": 5,
    "Nuremberg": 3,
    "Vienna": 15,
    "Salzburg": 5,
}

for city, size in cities.items():
    tree.add_node(city, size=size)

# Add non-leaf nodes (countries) - these will have optimizable radii
tree.add_node("Germany")
tree.add_node("Austria")

# Define containment relationships: (parent, child) means child is contained in parent
tree.add_edges_from([
    ("Germany", "Munich"),
    ("Germany", "Berlin"),
    ("Germany", "Stuttgart"),
    ("Germany", "Frankfurt"),
    ("Germany", "Nuremberg"),
    ("Austria", "Vienna"),
    ("Austria", "Salzburg"),
])

# Visualize the tree structure
nx.draw_networkx(tree, with_labels=True, node_color='lightblue', 
                 arrows=True, pos=nx.spring_layout(tree, seed=42))

In [ ]:
# Optimize the layout using the new function
pos, non_leaf_radii = bubblejax.optimize_circular_layout_with_enclosed_nodes(
    inclusion_tree=tree,
    optim_kwargs={"n_iters": 10000, "learning_rate": 1*1e-1}
)

# Visualize the optimized layout
_, ax = plt.subplots(figsize=(10, 10))

# Draw all nodes as circles
for node, node_xy in pos.items():
    # Determine radius: leaf nodes use their "size" attribute, non-leaf use optimized radii
    if tree.out_degree(node) == 0:  # leaf node
        radius = tree.nodes[node]["size"]
        color = 'lightcoral'  # cities in red
        alpha = 0.6
    else:  # non-leaf node
        radius = non_leaf_radii[node]
        color = 'lightblue'  # countries in blue
        alpha = 0.2
    
    circle = mpatches.Circle(node_xy, radius=radius, color=color, alpha=alpha, ec='black', linewidth=1)
    ax.add_patch(circle)
    ax.text(node_xy[0], node_xy[1], node, color="k", ha="center", va="center", 
            fontsize=10, weight='bold' if tree.out_degree(node) > 0 else 'normal')

# Set axis limits
max_radius = max(non_leaf_radii.values())
xmin = min(pos[node][0] - max_radius for node in pos)
xmax = max(pos[node][0] + max_radius for node in pos)
ymin = min(pos[node][1] - max_radius for node in pos)
ymax = max(pos[node][1] + max_radius for node in pos)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal")
ax.set_title("Optimized Bubble Layout: Cities Enclosed in Countries")
ax.grid(True, alpha=0.3)

print(f"Optimized radii for non-leaf nodes: {non_leaf_radii}")

In [ ]:
main_graph = nx.Graph()
main_graph.add_node("Munich", size=10)
main_graph.add_node("Frankfurt", size=5)
main_graph.add_node("Nuremberg", size=3)
main_graph.add_node("Berlin", size=25)
main_graph.add_node("Stuttgart", size=5)
main_graph.add_node("Salzburg", size=5)
main_graph.add_node("Vienna", size=15)
main_graph.add_edges_from([("Munich", "Frankfurt"), ("Munich", "Nuremberg"), ("Frankfurt", "Berlin")])
main_graph.add_edges_from([("Munich", "Stuttgart"), ("Stuttgart", "Berlin")])
main_graph.add_edges_from([("Munich", "Salzburg"), ("Salzburg", "Vienna")])
main_graph.add_edges_from([("Bratislava", "Vienna")])

n_add_nodes = 0
for i_node in range(1, n_add_nodes+1):
    main_graph.add_node(f"Node{i_node}", size=5)
    main_graph.add_edges_from([("Node"+str(i_node), "Munich")])


inclusion_tree = nx.DiGraph()
inclusion_tree.add_edge("Munich", "Germany")
inclusion_tree.add_edge("Berlin", "Germany")
inclusion_tree.add_edge("Stuttgart", "Germany")
inclusion_tree.add_edge("Frankfurt", "Germany")
inclusion_tree.add_edge("Salzburg", "Austria")
inclusion_tree.add_edge("Vienna", "Austria")
nx.draw_networkx(main_graph)

#pos = treescipy.optimize_tree_layout(inclusion_tree, node_dependent_min_deltas=False, min_delta=0.1, weight_edge_length=1)
_, ax =  plt.subplots();
nx.draw_networkx(inclusion_tree)

In [ ]:
pos, enclosing_node_radius_dict = bubblejax.optimize_circular_layout_with_enclosed_and_linked_nodes(graph=main_graph, inclusion_tree=inclusion_tree,
                                                         optim_kwargs={"n_iters": 10000, "learning_rate": 5*1e-1})
pos

In [ ]:
_, ax = plt.subplots()

for node, node_xy in pos.items():
    if node in main_graph.nodes:
        if "size" in main_graph.nodes[node]:
            radius = main_graph.nodes[node]["size"]
        else:
            radius = 1.0
    else:
        radius = enclosing_node_radius_dict[node]
    circle = mpatches.Circle(node_xy, radius=radius, alpha=0.2)
    ax.add_patch(circle)
    ax.text(node_xy[0], node_xy[1] + 0.9*radius, node, color="k", ha="center", va="center")
for edge in main_graph.edges:
    ax.plot(
        [pos[edge[0]][0], pos[edge[1]][0]],
        [pos[edge[0]][1], pos[edge[1]][1]],
        "k-",
    )
max_node_size = max(main_graph.nodes[node]["size"] for node in main_graph.nodes if "size" in main_graph.nodes[node])
max_node_size = max(enclosing_node_radius_dict.values())
xmin = min(pos[node][0] - max_node_size for node in pos)
xmax = max(pos[node][0] + max_node_size for node in pos)
ymin = min(pos[node][1] - max_node_size for node in pos)
ymax = max(pos[node][1] + max_node_size for node in pos)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
# equal axes
ax.set_aspect("equal")